# 01 — Data Infrastructure & Initial Exploration

**Purpose**: Foundation notebook for the wine-diversification correlation analysis.  
All downstream notebooks depend on the cleaned, saved datasets produced here.

## Sections
1. Environment setup
2. MotherDuck connection & schema discovery
3. Liv-ex index history (CSV)
4. Comparison assets from yfinance (S&P 500, FTSE 100, Gold)
5. Align to monthly frequency & save
6. Data quality assertions

## 1. Environment Setup

In [ ]:
import os
import warnings
import duckdb
import pandas as pd
import yfinance as yf
from pathlib import Path

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
REPO_ROOT = Path("__file__").resolve().parent.parent
DATA_DIR  = REPO_ROOT / "projects" / "correlation-diversification" / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

LIVEX_CSV        = DATA_DIR / "liv-ex_index_history.csv"
LIVEX_PARQUET    = DATA_DIR / "livex_indices_clean.parquet"
COMPARISON_PARQUET = DATA_DIR / "comparison_assets_monthly.parquet"

print("Data directory:", DATA_DIR)
print("motherduck_token present:", bool(os.getenv("motherduck_token")))

## 2. MotherDuck Connection & Schema Discovery

We query `information_schema.columns` for both tables to discover actual column names  
before touching any row data. We never assume column names.

**Tables**
- `winefi.ml.ml_unified_trades_tbvm` — unified trade data
- `winefi.mrt.mrt_unified_bids` — unified bid data

In [ ]:
con = duckdb.connect("md:")
print("Connected to MotherDuck")

In [ ]:
# Schema: ml_unified_trades_tbvm
trades_schema = con.execute("""
    SELECT
        column_name,
        data_type,
        is_nullable
    FROM information_schema.columns
    WHERE table_catalog = 'winefi'
      AND table_schema  = 'ml'
      AND table_name    = 'ml_unified_trades_tbvm'
    ORDER BY ordinal_position
""").df()

print("=== winefi.ml.ml_unified_trades_tbvm ===")
print(f"Rows in schema: {len(trades_schema)}")
display(trades_schema)

In [ ]:
# Schema: mrt_unified_bids
bids_schema = con.execute("""
    SELECT
        column_name,
        data_type,
        is_nullable
    FROM information_schema.columns
    WHERE table_catalog = 'winefi'
      AND table_schema  = 'mrt'
      AND table_name    = 'mrt_unified_bids'
    ORDER BY ordinal_position
""").df()

print("=== winefi.mrt.mrt_unified_bids ===")
print(f"Rows in schema: {len(bids_schema)}")
display(bids_schema)

In [ ]:
# Lightweight row-count + date-range preview (push aggregation into SQL)

# Identify the date column dynamically from the schema
trades_date_col = trades_schema[
    trades_schema["column_name"].str.lower().str.contains("date|time")
]["column_name"].iloc[0]

bids_date_col = bids_schema[
    bids_schema["column_name"].str.lower().str.contains("date|time")
]["column_name"].iloc[0]

print(f"Trades date column detected: {trades_date_col}")
print(f"Bids date column detected:   {bids_date_col}")

trades_summary = con.execute(f"""
    SELECT
        COUNT(*)                                    AS row_count,
        MIN(CAST(\"{trades_date_col}\" AS DATE))   AS min_date,
        MAX(CAST(\"{trades_date_col}\" AS DATE))   AS max_date
    FROM winefi.ml.ml_unified_trades_tbvm
    WHERE CAST(\"{trades_date_col}\" AS DATE) >= '2000-01-01'
""").df()

bids_summary = con.execute(f"""
    SELECT
        COUNT(*)                                  AS row_count,
        MIN(CAST(\"{bids_date_col}\" AS DATE))   AS min_date,
        MAX(CAST(\"{bids_date_col}\" AS DATE))   AS max_date
    FROM winefi.mrt.mrt_unified_bids
    WHERE CAST(\"{bids_date_col}\" AS DATE) >= '2000-01-01'
""").df()

print("\nTrades summary (from 2000):")
display(trades_summary)

print("Bids summary (from 2000):")
display(bids_summary)

## 3. Liv-ex Index History (CSV)

Load `projects/correlation-diversification/data/liv-ex_index_history.csv`,  
inspect its structure, document column names and date range, then save as parquet.

In [ ]:
livex_raw = pd.read_csv(LIVEX_CSV, index_col=0, parse_dates=True)

print("=== Liv-ex CSV — raw structure ===")
print(f"Shape:      {livex_raw.shape}")
print(f"Index type: {livex_raw.index.dtype}")
print(f"Date range: {livex_raw.index.min()} → {livex_raw.index.max()}")
print()
print("Columns and dtypes:")
print(livex_raw.dtypes)
print()
livex_raw.head()

In [ ]:
# Filter to 2000+ and resample to month-end
livex = livex_raw[livex_raw.index >= "2000-01-01"].copy()
livex.index = pd.to_datetime(livex.index)
livex_monthly = livex.resample("ME").last()

print("=== Liv-ex — monthly aligned ===")
print(f"Shape:      {livex_monthly.shape}")
print(f"Date range: {livex_monthly.index.min()} → {livex_monthly.index.max()}")
print()

# Column documentation
col_doc = pd.DataFrame({
    "column": livex_monthly.columns,
    "dtype":  livex_monthly.dtypes.values,
    "non_null": livex_monthly.notna().sum().values,
    "min": livex_monthly.min().values,
    "max": livex_monthly.max().values,
})
print("Column documentation:")
display(col_doc)

livex_monthly.to_parquet(LIVEX_PARQUET)
print(f"\nSaved → {LIVEX_PARQUET}")

## 4. Comparison Assets from yfinance

Pull monthly Close prices for:
- **S&P 500** (`^GSPC`)
- **FTSE 100** (`^FTSE`)
- **Gold** (`GC=F`)

Starting from 2000-01-01.

In [ ]:
TICKERS = {
    "sp500": "^GSPC",
    "ftse100": "^FTSE",
    "gold": "GC=F",
}
START_DATE = "2000-01-01"

frames = {}
for name, ticker in TICKERS.items():
    raw = yf.download(ticker, start=START_DATE, progress=False, auto_adjust=False)["Close"]
    if isinstance(raw, pd.DataFrame):
        raw = raw.squeeze()
    frames[name] = raw
    print(f"{name} ({ticker}): {len(raw)} daily rows, "
          f"{raw.index.min().date()} → {raw.index.max().date()}, "
          f"nulls={raw.isna().sum()}")

comparison_daily = pd.DataFrame(frames)
print(f"\nCombined shape: {comparison_daily.shape}")
comparison_daily.head()

## 5. Align All Time Series to Monthly Frequency

Resample everything to **last trading day of each calendar month** (month-end, `ME`).  
Merge Liv-ex indices with market comparison assets on the shared monthly index.

In [ ]:
# Resample comparison assets to month-end
comparison_monthly = comparison_daily.resample("ME").last()

# Merge with Liv-ex monthly
combined = comparison_monthly.join(livex_monthly, how="outer")
combined = combined[combined.index >= "2000-01-01"]

print("=== Combined monthly dataset ===")
print(f"Shape:      {combined.shape}")
print(f"Date range: {combined.index.min()} → {combined.index.max()}")
print()
print("Columns:")
for col in combined.columns:
    non_null = combined[col].notna().sum()
    print(f"  {col:<40} dtype={combined[col].dtype}  non_null={non_null}")

combined.head()

In [ ]:
combined.to_parquet(COMPARISON_PARQUET)
print(f"Saved combined monthly dataset → {COMPARISON_PARQUET}")
print(f"File size: {COMPARISON_PARQUET.stat().st_size / 1024:.1f} KB")

## 6. Data Quality Assertions

All assertions must pass before this notebook is considered complete.

In [ ]:
errors = []

# --- Liv-ex parquet ---
livex_check = pd.read_parquet(LIVEX_PARQUET)
if len(livex_check) < 100:
    errors.append(f"livex_indices_clean.parquet has only {len(livex_check)} rows (need > 100)")
if livex_check.index.min().year > 2000:
    errors.append(f"livex_indices_clean.parquet starts in {livex_check.index.min().year} (need <= 2000)")
if livex_check.index.max().year < 2015:
    errors.append(f"livex_indices_clean.parquet ends in {livex_check.index.max().year} (need >= 2015)")

# --- Combined monthly parquet ---
combined_check = pd.read_parquet(COMPARISON_PARQUET)
if len(combined_check) < 100:
    errors.append(f"comparison_assets_monthly.parquet has only {len(combined_check)} rows (need > 100)")
if combined_check.index.min().year > 2000:
    errors.append(f"comparison_assets_monthly.parquet starts in {combined_check.index.min().year} (need <= 2000)")
if combined_check.index.max().year < 2015:
    errors.append(f"comparison_assets_monthly.parquet ends in {combined_check.index.max().year} (need >= 2015)")

# --- yfinance columns present ---
for col in ["sp500", "ftse100", "gold"]:
    if col not in combined_check.columns:
        errors.append(f"Missing column '{col}' in comparison_assets_monthly.parquet")

if errors:
    for err in errors:
        print(f"FAIL: {err}")
    raise AssertionError("Data quality checks failed — see output above")
else:
    print("All data quality assertions PASSED.")
    print(f"  livex_indices_clean:      {len(livex_check)} rows, "
          f"{livex_check.index.min().date()} → {livex_check.index.max().date()}")
    print(f"  comparison_assets_monthly:{len(combined_check)} rows, "
          f"{combined_check.index.min().date()} → {combined_check.index.max().date()}")

## Summary

| Output file | Description | Frequency | Columns documented |
|-------------|-------------|-----------|--------------------|
| `livex_indices_clean.parquet` | Liv-ex index history, date-indexed | Monthly (ME) | Yes — see Section 3 |
| `comparison_assets_monthly.parquet` | S&P 500, FTSE 100, Gold + Liv-ex aligned | Monthly (ME) | Yes — see Section 5 |

**MotherDuck tables inspected**
- `winefi.ml.ml_unified_trades_tbvm` — column names documented in Section 2
- `winefi.mrt.mrt_unified_bids` — column names documented in Section 2

All prices are in **GBP** unless otherwise stated. Data filtered to 2000-01-01 onwards.